In [13]:
import pandas as pd
import joblib
import os
from pathlib import Path
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import joblib


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



In [2]:
print(os.getcwd())
project_root = Path.cwd().parent

/home/dread/Documents/Personal projects/SA-Water-Dam-Level-Predictor/notebooks


In [3]:
"""Feature_engineered file path"""
featured_data_csv = "../data/feature_engineered/engineered_data.csv"

In [4]:
"""Reads feature_engineered csv"""
df = pd.read_csv(featured_data_csv)
df = df.drop(["date", "lag_1_target_30"],axis=1)

"""Training size"""
train_size = int(len(df)*0.8)

"""80/20 train test split"""
train = df[:train_size]
test = df[train_size:]

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 4659 entries, 0 to 4658
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   water_level_mm               4659 non-null   float64
 1   year                         4659 non-null   int64  
 2   month                        4659 non-null   int64  
 3   lag_1                        4659 non-null   float64
 4   lag_7                        4659 non-null   float64
 5   lag_14                       4659 non-null   float64
 6   lag_28                       4659 non-null   float64
 7   lag_56                       4659 non-null   float64
 8   lag_84                       4659 non-null   float64
 9   rolling_mean_28              4659 non-null   float64
 10  rolling_mean_84              4659 non-null   float64
 11  weekly_sum                   4659 non-null   float64
 12  lag_7_weekly_sum             4659 non-null   float64
 13  week_over_week_pct           

In [5]:
X_train = train.drop("target_30", axis=1)
y_train = train["target_30"]

X_test = test.drop("target_30", axis=1)
y_test = test["target_30"]

In [6]:
"""Naive baseline model shift-1 baseline(Predict yesterdays values)"""
y_predict = y_test.shift(1).bfill()
print(y_predict.head())

3727    137.306239
3728    137.306239
3729    137.841807
3730    136.500022
3731    138.003437
Name: target_30, dtype: float64


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4659 entries, 0 to 4658
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   water_level_mm               4659 non-null   float64
 1   year                         4659 non-null   int64  
 2   month                        4659 non-null   int64  
 3   lag_1                        4659 non-null   float64
 4   lag_7                        4659 non-null   float64
 5   lag_14                       4659 non-null   float64
 6   lag_28                       4659 non-null   float64
 7   lag_56                       4659 non-null   float64
 8   lag_84                       4659 non-null   float64
 9   rolling_mean_28              4659 non-null   float64
 10  rolling_mean_84              4659 non-null   float64
 11  weekly_sum                   4659 non-null   float64
 12  lag_7_weekly_sum             4659 non-null   float64
 13  week_over_week_pct           

In [8]:
"""Baseline mean prediction shift-1 baseline(Predict yesterdays values)"""
baseline_mae = mean_absolute_error(y_test, y_predict)
print(f"Model MAE: {baseline_mae}")
print(f"y_test std:{y_test.std()}")  

"""Mean baseline (real benchmark)"""
mean_pred = [y_train.mean()] * len(y_test)
mean_mae = mean_absolute_error(y_test, mean_pred)
print(f"Real MAE baseline model: {mean_mae}")

Model MAE: 2.2329838332705534
y_test std:27.697306648887956
Real MAE baseline model: 23.32416017854055


In [10]:

day_30_model = XGBRegressor(
    n_estimators=1000,
    learning_rate = 0.1,
    max_depth = 6,
    min_child_weight = 1,
    subsample = 0.8,
    colsample_bytree = 0.8,
    n_jobs = -1,
    random_state = 0,
    early_stopping_rounds = 5
    )

day_30_model.fit(
    X_train, y_train,
    eval_set = [(X_test, y_test)],
    verbose = False
          )

xgboost_prediction = day_30_model.predict(X_test)
print(f"Mean absolute error: {mean_absolute_error(y_test, xgboost_prediction)}")

Mean absolute error: 13.715338339934993


In [15]:
joblib.dump(day_30_model, "../models/xgboost_rainfall_30,pkl")

['../models/xgboost_rainfall_30,pkl']